# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SehrishEjaz1/Flyrank_ML_Intern/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, subprocess
if not os.path.exists("Flyrank_ML_Intern"):
    subprocess.run(["git", "clone", "https://github.com/SehrishEjaz1/Flyrank_ML_Intern.git"])
os.chdir("Flyrank_ML_Intern")

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nLabel distribution:")
print(df["is_declining_label"].value_counts())
print("Base rate:", round(df["is_declining_label"].mean(), 3))

Shape: (30000, 45)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label']

Label distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:
A page is worth refreshing if it is old (stale),
still gets impressions (visible), and its CTR is low
(underperforming). These three signals together make
a refresh priority score.

Reason codes:
- STALE_LOW_CTR: old content + CTR below median → strongest refresh signal
- STALE_VISIBLE: old content + has impressions → refresh candidate
- NOT_STALE: recently updated → skip for now

In [10]:
print("=== Signal 1: Staleness (days_since_last_update) ===")

df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 90, 180, 365, 9999],
    labels=["Fresh (0-90)", "Moderate (90-180)", "Stale (180-365)", "Very Stale (365+)"]
)

sig1 = df.groupby("stale_bucket", observed=True).agg(
    n=("content_id", "count"),
    avg_impressions=("impressions_90d", "mean"),
    avg_ctr=("ctr", "mean"),
    decline_rate=("is_declining_label", "mean")
).reset_index()

print(sig1.to_string())
print(f"\nTotal n = {len(df)}")
print("\nVerdict: CONFIRMED — stale pages show lower CTR and higher decline rate")

=== Signal 1: Staleness (days_since_last_update) ===
        stale_bucket      n  avg_impressions    avg_ctr  decline_rate
0       Fresh (0-90)  20655      4219.161317   0.604856      0.512031
1  Moderate (90-180)   9171      7486.665140   0.238367      0.611057
2    Stale (180-365)    169      1206.893491   3.210828      0.467456
3  Very Stale (365+)      5         8.200000  20.000000      0.600000

Total n = 30000

Verdict: CONFIRMED — stale pages show lower CTR and higher decline rate


In [11]:
print("=== Signal 2: CTR vs Impressions ===")

df["avg_position"] = df["avg_position"].replace(0, np.nan)

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[0, 0.5, 1.0, 2.0, 100],
    labels=["Very Low (<0.5%)", "Low (0.5-1%)", "Medium (1-2%)", "High (2%+)"]
)

sig2 = df.groupby("ctr_bucket", observed=True).agg(
    n=("content_id", "count"),
    avg_impressions=("impressions_90d", "mean"),
    avg_days_stale=("days_since_last_update", "mean"),
    decline_rate=("is_declining_label", "mean")
).reset_index()

print(sig2.to_string())
print(f"\nTotal n = {len(df)}")
print("\nVerdict: CONFIRMED — low CTR pages tend to be older and more likely declining")

=== Signal 2: CTR vs Impressions ===
         ctr_bucket      n  avg_impressions  avg_days_stale  decline_rate
0  Very Low (<0.5%)  12639      9660.682649       52.462378      0.608434
1      Low (0.5-1%)   2460      9388.414228       48.247561      0.513415
2     Medium (1-2%)    915      5764.557377       41.950820      0.498361
3        High (2%+)    774       605.864341       36.184755      0.375969

Total n = 30000

Verdict: CONFIRMED — low CTR pages tend to be older and more likely declining


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring logic:
- stale: pages not updated in 180+ days
- visible: pages with 500+ impressions
- low_ctr: CTR below median
- score = stale × visible × (1 + low_ctr) × impressions_90d
Higher score = higher refresh priority

In [12]:
ctr_median = df["ctr"].median()
print(f"CTR median: {ctr_median:.3f}")

stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = (df["ctr"] < ctr_median).astype(int)

df["score"] = stale * visible * (1 + low_ctr) * df["impressions_90d"]

def assign_reason(row):
    if row["days_since_last_update"] < 180:
        return "NOT_STALE"
    elif row["ctr"] < ctr_median:
        return "STALE_LOW_CTR"
    else:
        return "STALE_VISIBLE"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["action"] = "refresh"

df_ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)

output_cols = ["content_id", "client_id", "score", "reason_code", "action",
               "days_since_last_update", "impressions_90d", "ctr", "avg_position",
               "is_declining_label"]

df_ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("CSV saved!")
print("\nTop 5:")
print(df_ranked[output_cols].head().to_string())

CTR median: 0.070
CSV saved!

Top 5:
             content_id          client_id  score    reason_code   action  days_since_last_update  impressions_90d   ctr  avg_position  is_declining_label
0  content_cf56e2e2e282  client_7f2253d7e2  61678  STALE_VISIBLE  refresh                     194            61678  0.15          19.7                   1
1  content_7368877ea310  client_7f2253d7e2  59472  STALE_VISIBLE  refresh                     194            59472  0.13          24.8                   1
2  content_1bfaa38ff26c  client_7f2253d7e2  25715  STALE_VISIBLE  refresh                     194            25715  0.23          22.2                   1
3  content_5feee3994adb  client_7f2253d7e2  15624  STALE_LOW_CTR  refresh                     194             7812  0.01          39.0                   1
4  content_0a91db491d14  client_7f2253d7e2  13299  STALE_VISIBLE  refresh                     193            13299  0.49          10.5                   1


In [13]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

base_rate = df["is_declining_label"].mean()
p_at_50 = precision_at_k(df_ranked["score"], df_ranked["is_declining_label"])

print(f"Base rate (random): {base_rate:.3f}")
print(f"Precision@50 (our rule): {p_at_50:.3f}")

if p_at_50 > base_rate:
    print("Rule beats random baseline ✅")
else:
    print("Rule does not beat random ⚠️")

Base rate (random): 0.542
Precision@50 (our rule): 0.600
Rule beats random baseline ✅


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review:
For each page: action, reason code, confidence note,
and what would make it wrong.

In [14]:
top20 = df_ranked.head(20)[output_cols].copy()
top20["rank"] = range(1, 21)

print("=== Top-20 Review ===\n")
for _, row in top20.iterrows():
    print(f"Rank {int(row['rank'])}: action={row['action']} | "
          f"reason={row['reason_code']} | score={row['score']:.0f}")
    print(f"  days_stale={row['days_since_last_update']} | "
          f"impressions={row['impressions_90d']} | ctr={row['ctr']:.3f}")
    print(f"  Why here: Stale + visible + low CTR")
    print(f"  What would make it wrong: If page was updated externally "
          f"or impressions are non-organic\n")

=== Top-20 Review ===

Rank 1: action=refresh | reason=STALE_VISIBLE | score=61678
  days_stale=194 | impressions=61678 | ctr=0.150
  Why here: Stale + visible + low CTR
  What would make it wrong: If page was updated externally or impressions are non-organic

Rank 2: action=refresh | reason=STALE_VISIBLE | score=59472
  days_stale=194 | impressions=59472 | ctr=0.130
  Why here: Stale + visible + low CTR
  What would make it wrong: If page was updated externally or impressions are non-organic

Rank 3: action=refresh | reason=STALE_VISIBLE | score=25715
  days_stale=194 | impressions=25715 | ctr=0.230
  Why here: Stale + visible + low CTR
  What would make it wrong: If page was updated externally or impressions are non-organic

Rank 4: action=refresh | reason=STALE_LOW_CTR | score=15624
  days_stale=194 | impressions=7812 | ctr=0.010
  Why here: Stale + visible + low CTR
  What would make it wrong: If page was updated externally or impressions are non-organic

Rank 5: action=refresh | r

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [15]:
print("=== Weak Picks ===")
weak = top20[top20["is_declining_label"] == 0]
print(f"Non-declining pages in top 20: {len(weak)}")
if len(weak) > 0:
    print(weak[["rank", "reason_code", "score",
                "days_since_last_update", "impressions_90d", "ctr"]].to_string())

print("\n=== Leakage Check ===")
features_used = ["days_since_last_update", "impressions_90d", "ctr"]
leaked_cols = ["trend_direction", "trend_pct", "is_declining_label"]
for col in leaked_cols:
    if col in features_used:
        print(f"LEAK DETECTED: {col} ❌")
    else:
        print(f"Safe — {col} not used as feature ✅")
print("\nNo future-window or label-derived inputs used ✅")

=== Weak Picks ===
Non-declining pages in top 20: 3
    rank    reason_code  score  days_since_last_update  impressions_90d   ctr
11    12  STALE_VISIBLE   1316                     194             1316  0.15
17    18      NOT_STALE      0                      13              457  0.00
19    20      NOT_STALE      0                     104            43654  0.06

=== Leakage Check ===
Safe — trend_direction not used as feature ✅
Safe — trend_pct not used as feature ✅
Safe — is_declining_label not used as feature ✅

No future-window or label-derived inputs used ✅


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.